# 🎮 LoL Win Predictor — Analyse Exploratoire (EDA)

**Projet fil rouge B3 IA & Data — Ynov Aix-en-Provence 2025/2026**  
Corin Deprez · Eliott Bellais

---

> **Question centrale :** À quel moment d'une partie de League of Legends le résultat devient-il statistiquement prévisible ?

Ce notebook explore les données de 1000+ matchs Master+ EUW pour comprendre quelles features prédisent le mieux la victoire, et à quelle minute la partie est « pliée ».

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_parquet('../data/processed/features.parquet')
print(f"Dataset : {len(df)} snapshots — {df['match_id'].nunique()} matchs uniques")
print(f"Blue win rate global : {df['blue_wins'].mean():.1%}")
df.head()

## 1. Distribution du dataset

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Snapshots par timestamp
snap_counts = df['game_time_minutes'].value_counts().sort_index()
axes[0].bar(snap_counts.index, snap_counts.values, color='steelblue', width=3)
axes[0].set_title('Snapshots par timestamp')
axes[0].set_xlabel('Minute de jeu')
axes[0].set_ylabel('Nombre de snapshots')
for i, (x, y) in enumerate(zip(snap_counts.index, snap_counts.values)):
    axes[0].text(x, y + 5, str(y), ha='center', fontsize=9)

# Winrate bleu par timestamp
wr_by_time = df.groupby('game_time_minutes')['blue_wins'].mean()
axes[1].bar(wr_by_time.index, wr_by_time.values * 100, color='royalblue', width=3)
axes[1].axhline(50, color='red', linestyle='--', alpha=0.6, label='50%')
axes[1].set_title('Win rate équipe bleue par timestamp')
axes[1].set_xlabel('Minute de jeu')
axes[1].set_ylabel('Win rate (%)')
axes[1].set_ylim(40, 60)
axes[1].legend()

# Distribution gold_diff à 20 min par résultat
df20 = df[df['game_time_minutes'] == 20]
axes[2].hist(df20[df20['blue_wins']==1]['gold_diff'], bins=30, alpha=0.6, color='royalblue', label='Victoire bleue')
axes[2].hist(df20[df20['blue_wins']==0]['gold_diff'], bins=30, alpha=0.6, color='tomato', label='Défaite bleue')
axes[2].axvline(0, color='black', linestyle='--', alpha=0.5)
axes[2].set_title('Distribution gold_diff à 20 min')
axes[2].set_xlabel('Gold diff (bleu - rouge)')
axes[2].legend()

plt.tight_layout()
plt.savefig('../docs/screenshots/01_dataset_overview.png', bbox_inches='tight')
plt.show()

## 2. Corrélations features × victoire

Quelle feature est la plus corrélée à la victoire ? On mesure le **point-biserial correlation** entre chaque feature et `blue_wins`.

In [ ]:
from src.features.build_features import FEATURE_COLS

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Corrélations globales
corrs = df[FEATURE_COLS + ['blue_wins']].corr()['blue_wins'].drop('blue_wins').sort_values()
colors = ['tomato' if c < 0 else 'royalblue' for c in corrs]
corrs.plot(kind='barh', ax=axes[0], color=colors)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Corrélation de chaque feature avec blue_wins')
axes[0].set_xlabel('Corrélation de Pearson')

# Heatmap inter-features
mask = np.triu(np.ones_like(df[FEATURE_COLS].corr(), dtype=bool))
sns.heatmap(df[FEATURE_COLS].corr(), ax=axes[1], mask=mask,
            cmap='RdBu_r', center=0, annot=True, fmt='.2f', annot_kws={'size': 7})
axes[1].set_title('Corrélations entre features')

plt.tight_layout()
plt.savefig('../docs/screenshots/02_correlations.png', bbox_inches='tight')
plt.show()

print("\nTop 5 features les plus corrélées à la victoire :")
print(corrs.abs().sort_values(ascending=False).head(5))

## 3. La question centrale — À quelle minute la partie est-elle pliée ?

On entraîne un modèle séparé par timestamp et on compare l'AUC. Plus l'AUC est haute, plus le modèle est confiant = plus la partie est lisible statistiquement.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler

timestamps = sorted(df['game_time_minutes'].unique())
auc_by_time = {}
n_by_time = {}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for t in timestamps:
    sub = df[df['game_time_minutes'] == t]
    X = sub[FEATURE_COLS]
    y = sub['blue_wins']
    n_by_time[t] = len(sub)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    lr = LogisticRegression(max_iter=500, random_state=42)
    scores = cross_val_score(lr, X_scaled, y, cv=cv, scoring='roc_auc')
    auc_by_time[t] = scores.mean()
    print(f"  {t:2d} min — AUC {scores.mean():.3f} ± {scores.std():.3f}  (n={len(sub)})")

# Le graphique storytelling principal
fig, ax = plt.subplots(figsize=(10, 5))
times = list(auc_by_time.keys())
aucs  = list(auc_by_time.values())

ax.plot(times, aucs, 'o-', color='royalblue', linewidth=2.5, markersize=9, zorder=3)
for t, a in zip(times, aucs):
    ax.annotate(f'{a:.3f}', (t, a), textcoords='offset points', xytext=(0, 12),
                ha='center', fontsize=11, fontweight='bold', color='royalblue')

ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Aléatoire (AUC=0.5)')
ax.fill_between(times, 0.5, aucs, alpha=0.1, color='royalblue')

ax.set_xlabel('Minute de jeu', fontsize=13)
ax.set_ylabel('AUC-ROC (cross-validation 5-folds)', fontsize=13)
ax.set_title('À quelle minute la partie est-elle statistiquement prévisible ?\n(plus l\'AUC est haute, plus le résultat est lisible)', fontsize=13)
ax.set_xticks(times)
ax.set_xticklabels([f'{t} min' for t in times], fontsize=11)
ax.set_ylim(0.45, 1.0)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/screenshots/03_auc_by_time.png', bbox_inches='tight')
plt.show()

## 4. Boxplots — Distribution des features par résultat

Pour chaque feature, on compare la distribution entre victoires et défaites. Visuellement évident pour la présentation.

In [ ]:
df20 = df[df['game_time_minutes'] == 20].copy()
df20['Résultat'] = df20['blue_wins'].map({1: 'Victoire bleue', 0: 'Défaite bleue'})

features_plot = ['gold_diff', 'kills_diff', 'towers_diff', 'dragons_diff',
                 'cs_diff', 'level_diff', 'barons_diff', 'kills_last_3min']
labels = ['Gold diff', 'Kills diff', 'Tours diff', 'Dragons diff',
          'CS diff', 'Level diff', 'Barons diff', 'Kills 3 dernières min']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
palette = {'Victoire bleue': 'royalblue', 'Défaite bleue': 'tomato'}

for i, (feat, label) in enumerate(zip(features_plot, labels)):
    sns.boxplot(data=df20, x='Résultat', y=feat, ax=axes[i],
                palette=palette, width=0.5, linewidth=1.2)
    axes[i].set_title(label, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].axhline(0, color='black', linestyle='--', alpha=0.4)
    axes[i].tick_params(axis='x', labelsize=9)

fig.suptitle('Distribution des features à 20 min — Victoire vs Défaite (équipe bleue)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../docs/screenshots/04_boxplots_20min.png', bbox_inches='tight')
plt.show()

## 5. Évolution des features moyennes au fil du temps

Comment les écarts entre équipes évoluent-ils sur une partie ? On trace la moyenne de chaque feature par timestamp, séparée victoires / défaites.

In [ ]:
key_features = ['gold_diff', 'kills_diff', 'towers_diff', 'dragons_diff']
labels_f = ['Gold diff', 'Kills diff', 'Tours diff', 'Dragons diff']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for i, (feat, label) in enumerate(zip(key_features, labels_f)):
    wins = df[df['blue_wins']==1].groupby('game_time_minutes')[feat].mean()
    losses = df[df['blue_wins']==0].groupby('game_time_minutes')[feat].mean()

    axes[i].plot(wins.index, wins.values, 'o-', color='royalblue', label='Victoire', linewidth=2)
    axes[i].plot(losses.index, losses.values, 's-', color='tomato', label='Défaite', linewidth=2)
    axes[i].axhline(0, color='black', linestyle='--', alpha=0.4)
    axes[i].fill_between(wins.index, wins.values, losses.values, alpha=0.08, color='royalblue')
    axes[i].set_title(label, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Minute')
    axes[i].set_xticks(timestamps)
    if i == 0:
        axes[i].legend()

fig.suptitle("Évolution des features moyennes — l'écart se creuse avec le temps",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/screenshots/05_feature_evolution.png', bbox_inches='tight')
plt.show()

## 6. Résumé EDA

| Feature | Corrélation victoire | Interprétation |
|---------|---------------------|----------------|
| `gold_diff` | ★★★★★ | Meilleur proxy économique, très séparant |
| `towers_diff` | ★★★★☆ | Contrôle de carte → pression directe |
| `kills_diff` | ★★★★☆ | Avantage combat, corrélé gold |
| `dragons_diff` | ★★★☆☆ | Important surtout > 20 min (dragon soul) |
| `cs_diff` | ★★★☆☆ | Farm passif, reflète l'économie |
| `level_diff` | ★★☆☆☆ | S'homogénéise avec le temps |
| `barons_diff` | ★★☆☆☆ | Rare, mais très décisif quand présent |
| `kills_last_3min` | ★★☆☆☆ | Momentum — utile en combinaison |

**Conclusion :** L'AUC augmente significativement entre 10 et 25 min. La partie est statistiquement lisible dès **15 min** pour les cas extrêmes, et **25 min** pour les parties serrées.